In [2]:
# ============================================================
# Mapa 3D de intensidad de campo magnético en una habitación
# ------------------------------------------------------------
# Toma mediciones puntuales de campo magnético (magnitud) tomadas en distintas posiciones de una habitación y genera:
#   1) un scatter 3D de las mediciones reales
#   2) un volumen 3D interpolado de la intensidad
# Ambos se exportan a HTML interactivo.
# ============================================================


In [29]:
import numpy as np
import pandas as pd
from scipy.interpolate import griddata
import plotly.graph_objects as go
import plotly.io as pio

# El CSV debe tener columnas de posición (en metros) y magnitud del campo (en µT). Ajustar la ruta al archivo propio.
df = pd.read_csv("datos_ejemplo.csv")

# Abrir los gráficos en el navegador (pantalla completa)
pio.renderers.default = "browser"


In [30]:
# Renombrar columnas a nombres sin espacios (más cómodos de usar).
df = df.rename(columns={
    "px en m": "px", "py en m": "py",
    "pz en m": "pz", "B en ut": "B"
})
# Quedarse solo con las 4 columnas útiles (descarta columnas vacías que a veces exporta la planilla, tipo "Unnamed: N").
df = df[["px", "py", "pz", "B"]]

# Arrays de posición (N×3) y de magnitud (N) que usa scipy.
pts = df[["px", "py", "pz"]].values
B = df["B"].values

In [31]:
# Chequeos rápidos: nombres de columna, cantidad de filas, y los valores más altos (útil para detectar errores de carga u outliers).
print(df.columns.tolist())
print(df.shape)
print(df.sort_values("B", ascending=False).head())

['px', 'py', 'pz', 'B']
(441, 4)
       px    py    pz      B
323  1.50  1.00  0.25  52.12
324  1.50  1.00  0.50  47.95
372  1.75  1.00  0.25  47.86
330  1.50  1.25  0.25  47.68
316  1.50  0.75  0.25  47.24


In [32]:
# Límites de color compartidos por ambos gráficos, para que los colores sean comparables entre el volumen y el scatter.
cmin = df.B.min()
cmax = df.B.max()

In [43]:
# ---------- scatter de las mediciones reales ----------
# Muestra solo los puntos efectivamente medidos, sin interpolación.
# Útil para ver la cobertura espacial y detectar outliers.
fig_puntos = go.Figure(go.Scatter3d(
    x=df.px, y=df.py, z=df.pz,
    mode="markers",
    marker=dict(
        size=5,
        color=df.B,
        colorscale="Turbo",
        cmin=cmin, cmax=cmax,     # misma escala que el volumen
        colorbar=dict(title="B (µT)"),
        showscale=True
    )
))
fig_puntos.update_layout(scene=dict(
    xaxis_title="x (m)", yaxis_title="y (m)", zaxis_title="z (m)",
    aspectmode="data"),
    title="Mediciones de campo magnético (puntos reales)")
fig_puntos.show()
fig_puntos.write_html("ejemplo_puntos_mediciones.html")

In [42]:
# ---------- volumen 3D interpolado ----------
# NOTA: un nº alto de nodos suaviza el volumen y lo hace ver más
# "continuo", pero sobreinterpola: genera estructura visual que no
# está respaldada por mediciones reales. Los nodos deberían aproximarse
# a (largo del eje / paso de medición). Acá se usan más solo con fines
# estéticos para la demo; con datos propios, mantenerlos acordes al muestreo.
xi = np.linspace(df.px.min(), df.px.max(), 20)
yi = np.linspace(df.py.min(), df.py.max(), 20)
zi = np.linspace(df.pz.min(), df.pz.max(), 20)
X, Y, Z = np.meshgrid(xi, yi, zi)

# Interpolación lineal (triangulación de Delaunay) en el interior del casco convexo de los puntos. Fuera de él deja NaN.
Bi = griddata(pts, B, (X, Y, Z), method="linear")
# Relleno de esos NaN de borde con vecino más cercano.
# OJO: esto es extrapolación, no dato medido. Comentar las 3 líneas siguientes para dejar los bordes vacíos.
Bi_fill = griddata(pts, B, (X, Y, Z), method="nearest")
mask = np.isnan(Bi)
Bi[mask] = Bi_fill[mask]

fig_volumen = go.Figure(go.Volume(
    x=X.flatten(), y=Y.flatten(), z=Z.flatten(),
    value=Bi.flatten(),
    isomin=cmin, isomax=cmax,     # rango de valores dibujado/coloreado
    opacity=0.10,                 # transparencia de las capas
    surface_count=130,             # nº de capas internas (más = más detalle)
    colorscale="Turbo",
    colorbar=dict(title="B (µT)"),
    caps=dict(x_show=False, y_show=False, z_show=False)  # sin tapas sólidas
))

#agregar marcadores de ciertos objetos reales de la habitación como la puerta
fig_volumen.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0],          # posición de la puerta
    mode="markers+text",
    text=["Puerta"],
    textposition="top center",
    marker=dict(size=6, color="black", symbol="square"),
    name="Puerta"
))

# lo agrego solo cuando quiero ver los datos crudos encima de la interpolación
# fig_volumen.add_trace(go.Scatter3d(
    #x=df.px, y=df.py, z=df.pz,
    #mode="markers",
    #marker=dict(size=3, color="black"),
    #name="mediciones reales"
#))
#fig_volumen.show()

fig_volumen.update_layout(scene=dict(
    xaxis_title="x (m)", yaxis_title="y (m)", zaxis_title="z (m)",
    aspectmode="data"),           # respeta proporciones reales del cuarto
    title="Intensidad de campo magnético (volumen interpolado)")
fig_volumen.show()
fig_volumen.write_html("ejemplo_campo_magnetico.html")